# CHR-99 / CHR-COLAB-003 Apriori threshold exploration and holdout evaluation

Use a fresh **free Colab CPU** runtime, with no accelerator and no Drive mount. Real preprocessed data and results are approved for the private GitHub repository. The current Jira assignee of CHR-99 is its sole official runner/publisher; other runs are exploratory. Verify the assignment before running and publishing. This notebook records a declared assignee, not authenticated Jira ownership, and never pushes results or credentials. Download the ZIPs locally for that assignee to review and publish manually.

Enter an explicit 40-character Git commit containing the audited `CHR-COLAB-002` artifacts and the database-independent ARM package initializer. Only the whitelisted Python sources, requirements, training CSV, and source manifest are fetched before exploration. No clone, original workbook, or holdout download occurs before selection is frozen. Thresholds have no defaults. Do not select or revise thresholds using holdout results.

## 1. Pin the inputs and retrieve training-only files

Use Python 3.12 or later and a least-privilege GitHub token with read access to repository contents. Enter it only in the hidden `getpass` prompt, never in a cell, URL, environment variable, or uploaded file. Tokens stay in memory for one retrieval phase and are not saved or logged. All requests use the authenticated `api.github.com` contents endpoint at the same commit. Restart with a new run if the commit, dataset version, or grid needs to change.

In [ ]:
import hashlib
import json
import os
import platform
import re
import sys
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from shutil import make_archive, which
from subprocess import run
from urllib.request import HTTPRedirectHandler, Request, build_opener
from uuid import uuid4
from google.colab import files

if 'run_root' in globals():
    raise RuntimeError('Restart the runtime to begin a new run.')
if sys.version_info < (3, 12):
    raise RuntimeError('Python 3.12 or later is required; select a compatible runtime.')
if input('Confirm free Colab CPU runtime with no accelerator (type YES): ').strip() != 'YES':
    raise ValueError('A free Colab CPU runtime is required.')
gpu = run(['nvidia-smi', '-L'], capture_output=True, text=True) if which('nvidia-smi') else None
if (gpu and gpu.returncode == 0 and gpu.stdout.strip()) or os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'):
    raise RuntimeError('Disable GPU/TPU acceleration and restart.')
repository = 'saiimonn/Chronos'
commit = input('Explicit 40-character Git commit SHA: ').strip().lower()
dataset_version = input('Preprocessed dataset version: ').strip()
publisher = input('Current CHR-99 assignee Jira account ID (check Jira first): ').strip()
if not re.fullmatch(r'[0-9a-f]{40}', commit):
    raise ValueError('A full 40-character commit SHA is required; branches and tags are not allowed.')
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]*', dataset_version):
    raise ValueError('Dataset version must be a single safe path component.')
if not re.fullmatch(r'[A-Za-z0-9:_-]+', publisher):
    raise ValueError('Enter the assignee Jira account ID, not a token or URL.')
if input('Confirm you are the current CHR-99 Jira assignee (type YES): ').strip() != 'YES':
    raise ValueError('Only the experiment ticket assignee can execute an official run.')

run_id = 'CHR-COLAB-003-' + uuid4().hex
run_root = Path('/content') / run_id
backend_dir = run_root / 'backend'
upload_dir = run_root / 'configuration'
preprocessing_dir = run_root / 'preprocessed'
for directory in (backend_dir, upload_dir, preprocessing_dir):
    directory.mkdir(parents=True, exist_ok=False)
dataset_prefix = f'dataset/preprocessed/{dataset_version}'
code_paths = (
    'backend/app/__init__.py',
    'backend/app/services/__init__.py',
    'backend/app/services/arm/__init__.py',
    'backend/app/services/arm/mining.py',
    'backend/app/services/arm/preprocessing.py',
    'backend/app/services/arm/rule_package.py',
    'backend/requirements.txt',
    'backend/scripts/colab_prototype/explore_apriori_holdout.py',
    'backend/scripts/colab_prototype/package_candidate_rules.py',
)
training_paths = tuple(f'{dataset_prefix}/{name}' for name in ('training_records.csv', 'source_manifest.json'))
holdout_path = f'{dataset_prefix}/holdout_records.csv'
destinations = {path: run_root / path for path in code_paths}
destinations.update({path: preprocessing_dir / path.rsplit('/', 1)[1] for path in (*training_paths, holdout_path)})
selection_lock = run_root / 'selection_lock.json'
lock_downloaded_sha256 = None
metadata = {
    'traceability': 'CHR-99 / CHR-COLAB-003',
    'run_id': run_id,
    'started_at_utc': datetime.now(timezone.utc).isoformat(),
    'repository': repository,
    'code_commit': commit,
    'dataset_commit': commit,
    'dataset_version': dataset_version,
    'official_publisher': {'source': 'jira_assignee', 'ticket': 'CHR-99', 'account_id': publisher},
    'publication_policy': 'Experiment ticket Jira assignee is sole official runner/publisher; manual review and publication only.',
    'environment': {
        'runtime': 'Google Colab', 'tier': 'free (operator confirmed)', 'accelerator': 'CPU',
        'python': sys.version, 'platform': platform.platform(),
        'machine': platform.machine(), 'processor': platform.processor(), 'cpu_count': os.cpu_count(),
        'memory_bytes': os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES'),
    },
    'retrieved_sha256': {}, 'completed_phases': [],
}

class NoRedirects(HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None

def fetch_github(paths):
    if (repository, commit, dataset_version) != (metadata['repository'], metadata['code_commit'], metadata['dataset_version']):
        raise RuntimeError('Pinned inputs changed; restart with a new run.')
    if any(path not in destinations for path in paths):
        raise ValueError('Retrieval is restricted to the explicit whitelist.')
    if holdout_path in paths:
        if not lock_downloaded_sha256 or not selection_lock.is_file():
            raise RuntimeError('Freeze and download the selection lock before retrieving holdout.')
        if hashlib.sha256(selection_lock.read_bytes()).hexdigest() != lock_downloaded_sha256:
            raise RuntimeError('The downloaded selection lock has changed.')
        run([*cli, 'verify',
             '--preprocessing-dir', str(preprocessing_dir),
             '--exploration-dir', str(exploration_dir),
             '--selected-thresholds', str(selected_thresholds),
             '--selection-lock', str(selection_lock)], cwd=backend_dir, check=True)
    token = getpass('GitHub read-only token (hidden; not saved): ').strip()
    if not token:
        raise ValueError('A token is required for the private repository.')
    try:
        opener = build_opener(NoRedirects())
        for path in paths:
            request = Request(f'https://api.github.com/repos/{repository}/contents/{path}?ref={commit}',
                              headers={'Authorization': f'Bearer {token}',
                                       'Accept': 'application/vnd.github.raw+json'})
            try:
                with opener.open(request, timeout=120) as response:
                    content = response.read()
            except Exception:
                raise RuntimeError(f'GitHub retrieval failed for {path}; check commit and read access.') from None
            destination = destinations[path]
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(content)
            metadata['retrieved_sha256'][path] = hashlib.sha256(content).hexdigest()
    finally:
        del token

def download_phase(phase):
    metadata['completed_phases'].append({'phase': phase, 'at_utc': datetime.now(timezone.utc).isoformat()})
    (run_root / 'run_metadata.json').write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')
    files.download(make_archive(str(Path('/content') / f'{run_id}-{phase}'), 'zip', run_root))

fetch_github((*code_paths, *training_paths))
run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements.txt'], cwd=backend_dir, check=True)
(run_root / 'installed_packages.txt').write_text(
    run([sys.executable, '-m', 'pip', 'freeze'], cwd=backend_dir, check=True, capture_output=True, text=True).stdout,
    encoding='utf-8',
)
print('Run ID:', run_id)

## 2. Upload the approved grid and explore training thresholds

Upload exactly `threshold_grid.json`, completed from the repository template by the researcher. Every value is a methodology decision; no grid values are supplied here. The CLI rejects holdout presence, verifies training against the source manifest, and evaluates all four streams. Review the comparison table and chart in the downloaded exploration ZIP before selecting thresholds. ZIPs contain real data and must remain private.

In [ ]:
if selection_lock.exists() or (preprocessing_dir / 'holdout_records.csv').exists():
    raise RuntimeError('Cannot explore after freezing or holdout access; start a new run.')
if (run_root / 'exploration').exists():
    raise RuntimeError('Exploration already exists; start a new run to change the grid.')
uploaded = files.upload()
if set(uploaded) != {'threshold_grid.json'}:
    raise ValueError('Upload exactly threshold_grid.json; no data ZIP or workbook.')
threshold_grid = upload_dir / 'threshold_grid.json'
threshold_grid.write_bytes(uploaded['threshold_grid.json'])
del uploaded
exploration_dir = run_root / 'exploration'
cli = [sys.executable, '-m', 'scripts.colab_prototype.explore_apriori_holdout']
run([*cli, 'explore',
     '--preprocessing-dir', str(preprocessing_dir),
     '--threshold-grid', str(threshold_grid),
     '--output-dir', str(exploration_dir)], cwd=backend_dir, check=True)
download_phase('exploration')

## 3. Freeze and download the training-only selection

Review only training results, record the selection rationale for publisher review, and upload `selected_thresholds.json` completed from the repository template. Each selected combination must have appeared in exploration. The `freeze` CLI creates the immutable selection lock. Download that lock and the frozen-selection ZIP before continuing. Do not change the selection after holdout access.

In [ ]:
if selection_lock.exists() or (preprocessing_dir / 'holdout_records.csv').exists():
    raise RuntimeError('Selection already frozen or holdout accessed; do not replace this selection.')
rationale = input('Training-only selection rationale for publisher review: ').strip()
if not rationale:
    raise ValueError('Record a training-only selection rationale.')
uploaded = files.upload()
if set(uploaded) != {'selected_thresholds.json'}:
    raise ValueError('Upload exactly selected_thresholds.json.')
selected_thresholds = upload_dir / 'selected_thresholds.json'
selected_thresholds.write_bytes(uploaded['selected_thresholds.json'])
del uploaded
run([*cli, 'freeze',
     '--exploration-dir', str(exploration_dir),
     '--selected-thresholds', str(selected_thresholds),
     '--selection-lock', str(selection_lock)], cwd=backend_dir, check=True)
metadata['selection_rationale'] = rationale
metadata['selection_lock_sha256'] = hashlib.sha256(selection_lock.read_bytes()).hexdigest()
files.download(str(selection_lock))
download_phase('frozen-selection')

## 4. Retrieve holdout at the same commit and evaluate the locked selection

Confirm that the lock download arrived before authorizing holdout retrieval. The `verify` CLI checks frozen inputs and imported ARM source hashes before any holdout request. Only then does the notebook fetch `holdout_records.csv`, using a fresh hidden token prompt and the original commit. Evaluation repeats the checks before reading holdout. A blank attainment means no applicable rule, not zero evidence. Inspect actual CSVs and charts in the evaluation ZIP; this notebook contains no saved outputs or invented official metrics. Free CPU runs may take time for large grids.

In [ ]:
if not selection_lock.is_file() or 'selection_lock_sha256' not in metadata:
    raise RuntimeError('Run the freeze phase and download its lock first.')
if input('Confirm selection_lock.json downloaded locally (type DOWNLOADED): ').strip() != 'DOWNLOADED':
    raise RuntimeError('Download the frozen lock before holdout access.')
lock_downloaded_sha256 = metadata['selection_lock_sha256']
metadata['lock_download_confirmed_at_utc'] = datetime.now(timezone.utc).isoformat()
if (run_root / 'evaluation').exists():
    raise RuntimeError('Evaluation already exists; preserve the results for publisher review.')
fetch_github((holdout_path,))
evaluation_dir = run_root / 'evaluation'
run([*cli, 'evaluate',
     '--preprocessing-dir', str(preprocessing_dir),
     '--exploration-dir', str(exploration_dir),
     '--selected-thresholds', str(selected_thresholds),
     '--selection-lock', str(selection_lock),
     '--output-dir', str(evaluation_dir)], cwd=backend_dir, check=True)
download_phase('evaluation')

## 5. Package the reviewed candidate for offline import

Copy `faculty_crosswalk.template.csv` locally, replace its example row with every retained faculty in `faculty_identity_map.csv`, and add each reviewed local PostgreSQL `faculty_id`. Upload exactly that completed `faculty_crosswalk.csv`. This step reads only local runtime files, verifies the CHR-99 hashes, and downloads one self-contained `candidate_rule_set.json`; it does not connect to PostgreSQL or any cloud database.

In [ ]:
if not (evaluation_dir / 'evaluation_manifest.json').is_file():
    raise RuntimeError('Complete the frozen holdout evaluation first.')
candidate_package = run_root / 'candidate_rule_set.json'
if candidate_package.exists():
    raise RuntimeError('Candidate package already exists; preserve it and start a new run for changes.')
uploaded = files.upload()
if set(uploaded) != {'faculty_crosswalk.csv'}:
    raise ValueError('Upload exactly faculty_crosswalk.csv.')
faculty_crosswalk = upload_dir / 'faculty_crosswalk.csv'
faculty_crosswalk.write_bytes(uploaded['faculty_crosswalk.csv'])
del uploaded
package_cli = [sys.executable, '-m', 'scripts.colab_prototype.package_candidate_rules']
run([*package_cli,
     '--preprocessing-dir', str(preprocessing_dir),
     '--evaluation-dir', str(evaluation_dir),
     '--faculty-crosswalk', str(faculty_crosswalk),
     '--output', str(candidate_package)], cwd=backend_dir, check=True)
files.download(str(candidate_package))

## Handoff to CHR-100 / CHR-COLAB-004

The CHR-99 Jira assignee reviews the ZIP's `run_metadata.json`, installed packages, retrieved source hashes, source manifest, approved grid, rationale, selection lock, and CLI manifests before manually publishing official artifacts to the private repository. Verify assignment again before publication; record ownership changes in Jira rather than relabeling past evidence. No automatic push or Drive persistence occurs.

`selected_rules.csv` remains a research artifact, not a database import file. Only the validated `candidate_rule_set.json` is transferred manually to the local Chronos host. Run `python -m app.cli.import_rule_set --artifact candidate_rule_set.json --dry-run` before the real import. The command validates faculty, subject, room, and training-term references against local PostgreSQL and always creates a draft. Historical rules remain soft guidance and require explicit Department Administrator validation and activation.